# Clase 8 — Auxiliares · el gimnasio

8 ejercicios cortos sobre *Construir MatchingEngine*: drills para ganar soltura con las primitivas y profundizaciones opcionales.

⏱️ 🟢 núcleo ~6 min · 🔵 si vamos bien +6 min · 🟣 bonus +5 min.

### El gimnasio

Drills cortos para automatizar las primitivas de Python con datos de mercado, más una profundización final. Mismo formato de siempre: escribe tu código, ejecuta la **✅ comprobación plegada** (`Shift+Enter`) y, si te atascas, abre **💡 Ver solución**. Ninguno debería llevarte más de un par de minutos. No hacen falta para seguir el curso — pero te hacen rápido.

**Dosis mínima** = todo lo marcado **🟢 núcleo**: el calentamiento entero y los dos primeros drills de cada bloque. Lo **🔵 si vamos bien** y lo **🟣 bonus**, para volver otro día.

---

## 🏋️ Gimnasio · Calentamiento — repaso exprés de L7

Dos lecturas rápidas del libro real antes de dispararle.

### C1. Foto exprés

<sub>🟢 núcleo · ~1 min</sub>

Del primer snapshot, guarda `mid0` y `spread0`.

<sub>practicas: repaso: mid y spread</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import OrderBook
book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
mid0 = None
spread0 = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert mid0 is not None, '⏸ mid0 sigue en None: completa el ejercicio antes de validar'
assert spread0 is not None, '⏸ spread0 sigue en None: completa el ejercicio antes de validar'
assert abs(mid0 - book.mid) < 1e-9 and abs(spread0 - book.spread) < 1e-9
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
mid0 = book.mid
spread0 = book.spread
```

</details>

### C2. Liquidez exprés

<sub>🟢 núcleo · ~1 min</sub>

¿Cuánto se puede comprar contra los 3 primeros asks? Guarda `d3`.

<sub>practicas: repaso: depth</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import OrderBook
book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
from exchange.orders import Side
d3 = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert d3 is not None, '⏸ d3 sigue en None: completa el ejercicio antes de validar'
assert abs(d3 - book.depth(Side.SELL, 3)) < 1e-9
print(f'ok  d3={d3:.3f}')

<details>
<summary>💡 Ver solución</summary>

```python
d3 = book.depth(Side.SELL, 3)
```

</details>

---

## 🏋️ Gimnasio · Bloque 1 — Fills, barridos y el peaje

La aritmética del matching, contra el motor real.

### A1. La orden pequeña no barre

<sub>🟢 núcleo · ~2 min</sub>

Envía una market buy de la MITAD del mejor ask. Comprueba que sale UN fill al precio `best_ask`. Guarda `fills`.

<sub>practicas: market vs nivel 1</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import OrderBook
book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
from exchange.matching import MatchingEngine
from exchange.orders import Order, OrderType
size = book.asks[0].size / 2
best_ask_antes = book.asks[0].price  # process() consume el libro: captura antes
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert fills is not None, '⏸ fills sigue en None: completa el ejercicio antes de validar'
assert len(fills) == 1, 'cabe entera en el nivel 1: un solo fill'
assert abs(fills[0].price - best_ask_antes) < 1e-9, 'se llena AL precio del mejor ask'
assert abs(fills[0].size - size) < 1e-9
print('ok ->', fills[0])

<details>
<summary>💡 Ver solución</summary>

```python
engine = MatchingEngine()
fills = engine.process(Order('BTCUSDT', 'buy', size, order_type=OrderType.MARKET), book)
```

</details>

### A2. Precio efectivo

<sub>🟢 núcleo · ~2 min</sub>

Con los fills dados (tuplas precio-tamaño), calcula `eff`, el precio medio ponderado.

<sub>practicas: media ponderada de fills</sub>

In [ ]:
fills = [(100010.0, 0.8), (100025.0, 0.5), (100040.0, 0.7)]
eff = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eff is not None, '⏸ eff sigue en None: completa el ejercicio antes de validar'
assert abs(eff - 100024.25) < 1e-2, 'suma p*s / suma s'
print(f'ok  eff={eff:.2f}')

<details>
<summary>💡 Ver solución</summary>

```python
eff = sum(p * s for p, s in fills) / sum(s for p, s in fills)
```

</details>

### A3. El slippage con signo

<sub>🔵 si vamos bien · ~2 min</sub>

Compraste con `eff = 100024.25` y el mid era `100000.0`. Guarda `slip` (positivo = pagaste de más) y `slip_bps`.

<sub>practicas: eff vs mid</sub>

In [ ]:
eff = 100024.25
mid = 100000.0
slip = None
slip_bps = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert slip is not None, '⏸ slip sigue en None: completa el ejercicio antes de validar'
assert slip_bps is not None, '⏸ slip_bps sigue en None: completa el ejercicio antes de validar'
assert abs(slip - 24.25) < 1e-9
assert abs(slip_bps - 2.425) < 1e-3
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
slip = eff - mid
slip_bps = slip / mid * 10000
```

</details>

### A4. El barrido, medido

<sub>🔵 si vamos bien · ~2 min</sub>

Dispara una market buy que se coma los 2 primeros niveles ask enteros. Comprueba que salen 2 fills y guarda `eff`.

<sub>practicas: el motor contra el libro real</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import OrderBook
book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
from exchange.matching import MatchingEngine
from exchange.orders import Order, OrderType
size = book.asks[0].size + book.asks[1].size
mid_antes = book.mid  # process() consume el libro: captura antes
eff = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eff is not None, 'calcula el precio efectivo de los fills'
assert eff > mid_antes, 'compras barriendo: el efectivo queda POR ENCIMA del mid'
print(f'ok  eff={eff:.2f}  (mid era {mid_antes:.2f})')

<details>
<summary>💡 Ver solución</summary>

```python
engine = MatchingEngine()
fills = engine.process(Order('BTCUSDT', 'buy', size, order_type=OrderType.MARKET), book)
eff = sum(f.price * f.size for f in fills) / sum(f.size for f in fills)
```

</details>

### A5. FOK: todo o nada

<sub>🔵 si vamos bien · ~2 min</sub>

Envía una FOK de compra MÁS GRANDE que toda la liquidez ask visible, a precio del último nivel. Comprueba que devuelve 0 fills y que el libro queda intacto.

<sub>practicas: el contrato FOK</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange.book import OrderBook
book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
from exchange.matching import MatchingEngine
from exchange.orders import Order, OrderType
total_ask = sum(lv.size for lv in book.asks)
ask_antes = book.best_ask
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert fills is not None, '⏸ fills sigue en None: completa el ejercicio antes de validar'
assert fills == []
assert abs(book.best_ask - ask_antes) < 1e-9, 'FOK imposible no toca el libro'
print('ok — todo o nada')

<details>
<summary>💡 Ver solución</summary>

```python
engine = MatchingEngine()
fills = engine.process(Order('BTCUSDT', 'buy', total_ask * 2, price=book.asks[-1].price, order_type=OrderType.FOK), book)
```

</details>

---

## 🏋️ Para terminar — profundización

El slippage como función del tamaño, sistematizado.

### A6. Slippage por tamaño

<sub>🟣 bonus · ~5 min</sub>

Para tamaños 0.1, 1.0 y 5.0, calcula el precio efectivo de una market buy. Guarda la lista `eff_prices`. Debe ser creciente (más tamaño, peor precio).

<sub>practicas: coste vs tamaño</sub>

In [ ]:
import csv
import os
import exchange

_data_path = os.path.join(os.path.dirname(exchange.__file__), '_data', 'btc_lob_snapshots.csv')
with open(_data_path, newline='') as _f:
    row = {k: float(v) for k, v in next(csv.DictReader(_f)).items()}
from exchange import OrderBook, MatchingEngine, Order, Side, OrderType
eff_prices = []

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert len(eff_prices) == 3, '⏸ llena eff_prices con los 3 precios efectivos antes de validar'
assert eff_prices[0] <= eff_prices[1] <= eff_prices[2], 'más tamaño = peor precio'
print('ok', [round(p,1) for p in eff_prices])

<details>
<summary>💡 Ver solución</summary>

```python
eng = MatchingEngine()
eff_prices = []
for size in (0.1, 1.0, 5.0):
    book = OrderBook.from_snapshot('BTCUSDT', row, depth=10)
    fills = eng.process(Order('BTCUSDT', Side.BUY, size, order_type=OrderType.MARKET), book)
    eff_prices.append(sum(f.price*f.size for f in fills)/sum(f.size for f in fills))
```

</details>

## Fin de los auxiliares

Vuelve al cuaderno principal cuando quieras.